# 02 — Baseline Logistic Regression

**Goal:** Establish a transparent, reproducible baseline that any fancier model has to beat. Logistic regression also gives interpretable coefficients — useful for sanity-checking that the model agrees with the EDA findings.

**Why a baseline matters:** if XGBoost only beats LR by 0.01 AUC, the extra complexity is rarely worth it. If it beats LR by 0.05+, the nonlinearities and interactions are worth the extra training/inference cost.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_clean
from src.features import make_features, train_test_split_scaled
from src.model import train_logistic_regression
from src.evaluation import (
    classification_metrics,
    feature_importance_df,
    plot_confusion,
    plot_roc,
)

pd.set_option("display.max_rows", 40)

## 1. Load, featurize, split

Scaling is on for logistic regression — distance-based optimizers converge faster and the coefficients become comparable in magnitude.

In [ ]:
df = load_clean()
X, y = make_features(df)
X_train, X_test, y_train, y_test = train_test_split_scaled(X, y, scale=True)

print(f"Train: {X_train.shape}, positive rate = {y_train.mean():.1%}")
print(f"Test:  {X_test.shape}, positive rate = {y_test.mean():.1%}")
print(f"Features: {X_train.shape[1]}")

## 2. Fit

Default settings: L2, liblinear solver, `class_weight="balanced"` to offset the 73/27 split.

In [ ]:
model = train_logistic_regression(X_train, y_train)
model

## 3. Evaluate on the held-out test set

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = classification_metrics(y_test, y_pred, y_proba)
pd.Series(metrics).round(3).to_frame("logistic_regression")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_roc(y_test, y_proba, label="Logistic Regression", ax=axes[0])
plot_confusion(y_test, y_pred, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Coefficients — which features drive the prediction?

Positive coefficient → pushes prediction toward **churn**. Negative → pushes toward **stay**. Because we scaled the numeric inputs, the magnitudes are directly comparable.

In [ ]:
coefs = feature_importance_df(model, X_train.columns.tolist())
coefs.head(15)

In [ ]:
# Top 10 by magnitude, signed
top = coefs.head(10).iloc[::-1]  # reverse for horizontal bar
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["crimson" if v > 0 else "steelblue" for v in top["coefficient"]]
ax.barh(top["feature"], top["coefficient"], color=colors)
ax.axvline(0, color="k", linewidth=0.8)
ax.set_title("Top 10 logistic-regression coefficients (by |magnitude|)")
ax.set_xlabel("coefficient")
plt.tight_layout()
plt.show()

## 5. Takeaways

- **Expected top drivers** (confirmed if above chart shows them): short `tenure`, month-to-month `Contract`, fiber `InternetService`, electronic-check `PaymentMethod`, lack of `OnlineSecurity` / `TechSupport`.
- **Baseline AUC** in the 0.83–0.85 range is typical for this dataset — anything XGBoost delivers needs to clear that bar meaningfully.
- **Recall on churners matters more than accuracy** — we'll revisit the decision threshold in the Week 2 cost-sensitive notebook.